# Checkign the device

In [1]:
import torch

print("Is a ROCm-GPU detected? ", torch.cuda.is_available())
print("How many ROCm-GPUs are detected? ", torch.cuda.device_count())

Is a ROCm-GPU detected?  True
How many ROCm-GPUs are detected?  1


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:736: UserWarning: Can't initialize amdsmi - Error code: 34
  warnings.warn(f"Can't initialize amdsmi - Error code: {e.err_code}")


# Library setup

In [2]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
import json
import os

/opt/conda/envs/py_3.12/lib/python3.12/site-packages/redis/connection.py:77: UserWarning: redis-py works best with hiredis. Please consider installing
  warnings.warn(msg)


# Dataset preparation

In [3]:
os.getcwd()

'/workspace/opendrive_generation'

In [4]:
# Load dataset from JSONL
dataset = load_dataset(
    "json",
    data_files="/workspace/opendrive_generation/xodr_generated_scenarios_20250624_161625/scenario_metadata.jsonl",
    split="train",
)

# Shuffle for training variety
dataset = dataset.shuffle(seed=42)


# Combine prompt and response into a training text field
def format_example(example):
    prompt = example.get("prompt", "").strip()

    if "script_path" not in example:
        raise RuntimeError("No script_path in the json entry.")

    script_path = example.get("script_path", "")
    code = ""

    with open(f"{os.getcwd()}/{script_path}", "r") as code_file:
        code = code_file.read().strip()

    response = example.get("response", "").strip()
    return {
        "text": f"### Prompt:\n{prompt}\n\n### Response:\n{code}",
        "prompt": prompt,
        "response": code,
    }


# Apply mapping
dataset = dataset.map(format_example)

# Example check
print(dataset[0]["text"])

### Prompt:
Start with a straight road and gently curve it away using a spiral shape.

### Response:
from scenariogeneration import xodr, prettyprint, ScenarioGenerator

class Scenario(ScenarioGenerator):
    def __init__(self):
        super().__init__()

    def road(self):
        road = xodr.create_road(
            [xodr.Line(30), xodr.Spiral(-0.00001, -0.035, 200)],
            id=2,
            left_lanes=2,
            right_lanes=2
        )
        odr = xodr.OpenDrive("line_spiral_combo")
        odr.add_road(road)
        odr.adjust_roads_and_lanes()
        return odr

if __name__ == "__main__":
    sce = Scenario()
    prettyprint(sce.road().get_element())
    sce.generate(".")


# Model

In [ ]:
# Load base model to GPU memory.
device = "cuda:0"
model_name = (
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # or 'mistralai/Mistral-7B-Instruct-v0.2'
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto", load_in_4bit=True, trust_remote_code=True
)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
/opt/conda/envs/py_3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:736: UserWarning: Can't initialize amdsmi - Error code: 34
  warnings.warn(f"Can't initialize amdsmi - Error code: {e.err_code}")


g++ (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [ ]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [ ]:
def tokenize(batch):
    prompts = [f"### Prompt:\n{p}\n\n### Response:\n" for p in batch["prompt"]]
    responses = batch["response"]
    full_texts = [prompt + response for prompt, response in zip(prompts, responses)]

    encodings = tokenizer(
        full_texts, padding="max_length", truncation=True, max_length=512
    )

    # Mask the prompt section in the labels
    labels = []
    for i in range(len(prompts)):
        label = encodings["input_ids"][i].copy()
        prompt_len = len(tokenizer(prompts[i])["input_ids"])
        label[:prompt_len] = [-100] * prompt_len
        labels.append(label)

    encodings["labels"] = labels
    return encodings


tokenized = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

# Fine Tuning loop

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False  # we're doing causal LM, not masked LM
)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir llm_xodr_finetuned/runs

ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
TensorFlow installation not found - running with reduced feature set.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

Traceback (most recent call last):
  File "/opt/conda/envs/py_3.12/bin/tensorboard", line 8, in <module>
    sys.exit(run_main())
             ^^^^^^^^^^
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/tensorboard/main.py", line 46, in run_main
    app.run(tensorboard.main, flags_parser=tensorboard.configure)
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/absl/app.py", line 316, in run
    _run_main(main, args)
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/absl/app.py", line 261, in _run_main
    sys.exit(main(argv))
             ^^^^^^^^^^
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/tensorboard/p

In [ ]:
from transformers import TrainerCallback
import torch

class PromptLoggingCallback(TrainerCallback):
    def __init__(self, tokenizer, dataset, interval=50):
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.interval = interval

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.interval == 0:
            model = kwargs['model']
            prompt = self.dataset[state.global_step % len(self.dataset)]['prompt']
            formatted = f"### Prompt:\n{prompt}\n\n### Response:\n"
            input_ids = self.tokenizer(formatted, return_tensors="pt").to(model.device)
            
            model.eval()
            with torch.no_grad():
                outputs = model.generate(
                    **input_ids,
                    max_new_tokens=300,
                    do_sample=False,
                    temperature=0.0,
                    pad_token_id=self.tokenizer.eos_token_id
                )
            decoded = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            response_only = decoded.replace(formatted, "").strip()

            print("\n" + "=" * 80)
            print(f"Step {state.global_step}")
            print("📥 Prompt:\n", prompt)
            print("📤 Model Output:\n", response_only)
            print("=" * 80 + "\n")

In [ ]:
from transformers import TrainingArguments, Trainer
prompt_logger = PromptLoggingCallback(tokenizer=tokenizer, dataset=dataset, interval=50)

args = TrainingArguments(
    output_dir="llm_xodr_finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    logging_steps=50,
    num_train_epochs=3,
    save_strategy="epoch",
    fp16=True,
    report_to="tensorboard",
    label_names=["labels"],  # <-- this line solves the warning
    callbacks=[prompt_logger]  # 👈 Add it here
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# trainer = Trainer(model=model, args=args, train_dataset=tokenized, tokenizer=tokenizer)
trainer.train()

/tmp/ipykernel_68694/3532229575.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/opt/conda/envs/py_3.12/lib/python3.12/site-packages/bitsandbytes/nn/modules.py:438: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(
/opt/conda/envs/py_3.12/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:54: UserWarning: Using AOTriton backend for Efficient Attention forward... (Triggered internally at /var/lib/jenkins/pytorch/aten/src/ATen/native/transformers/hip/attention.hip:1180.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/opt/conda/envs/py_3.12/lib/python3.12/site-packages/torch/autograd/graph.py:823: UserWarning: Using AOTriton backend for Efficient Attention backward... (Triggered internally at /var/lib/jenkins

Step,Training Loss
50,0.961000
100,0.287300


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:736: UserWarning: Can't initialize amdsmi - Error code: 34
  warnings.warn(f"Can't initialize amdsmi - Error code: {e.err_code}")
/opt/conda/envs/py_3.12/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:54: UserWarning: Using AOTriton backend for Flash Attention forward... (Triggered internally at /var/lib/jenkins/pytorch/aten/src/ATen/native/transformers/hip/flash_attn/flash_api.h:267.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/opt/conda/envs/py_3.12/lib/python3.12/site-packages/torch/autograd/graph.py:823: UserWarning: Using AOTriton backend for Flash Attention backward... (Triggered internally at /var/lib/jenkins/pytorch/aten/src/ATen/native/transformers/hip/flash_attn/flash_api.h:452.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


KeyboardInterrupt: 

# Inference testing

In [ ]:
model.eval()
prompt = "Make a road that curves gently to the right."
inputs = tokenizer(f"### Prompt:\n{prompt}\n\n### Response:\n", return_tensors="pt").to(
    "cuda"
)
outputs = model.generate(**inputs, max_new_tokens=1000)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/opt/conda/envs/py_3.12/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:54: UserWarning: Using AOTriton backend for Flash Attention forward... (Triggered internally at /var/lib/jenkins/pytorch/aten/src/ATen/native/transformers/hip/flash_attn/flash_api.h:267.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


### Prompt:
Make a road that curves gently to the right.

### Response:
The road curves gently to the right.
